# Semana 28 · ¿Cómo funciona un telescopio?

Este cuaderno acompaña a la sesión de proyecto de la semana 28: la ley de Snell en un prisma, la ecuación de las lentes delgadas, una calculadora de instrumentos y el disco de Airy.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. La ley de Snell en un prisma

$$n_1 \sin\theta_1 = n_2 \sin\theta_2.$$

El índice de refracción del vidrio depende un poco del color. Una aproximación habitual es la **fórmula de Cauchy**, $n(\lambda) = A + B/\lambda^2$; con $A = 1{,}5046$ y $B = 4200$ nm² se parece al vidrio óptico BK7. Seguimos un rayo a través de un prisma de 60° para tres colores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def n_vidrio(lam_nm):
    return 1.5046 + 4200 / lam_nm**2

def snell(theta1, n1, n2):
    """Ángulo de salida (radianes) o None si hay reflexión total."""
    s = n1 * np.sin(theta1) / n2
    return None if abs(s) > 1 else np.arcsin(s)

for lam in [450, 550, 650]:
    print(f"λ = {lam} nm: n = {n_vidrio(lam):.4f}")

Para un prisma de ángulo $A$, si el rayo entra con ángulo $\theta_1$, dentro se dobla a $\theta_2$; llega a la segunda cara con $\theta_3 = A - \theta_2$ y sale con $\theta_4$. La **desviación** total es $\theta_1 + \theta_4 - A$.

In [ ]:
A = np.radians(60)
th1 = np.radians(np.linspace(30, 80, 200))
plt.figure(figsize=(8, 4.5))
for lam, color in [(450, "royalblue"), (550, "green"), (650, "red")]:
    n = n_vidrio(lam)
    desv = []
    for t in th1:
        t2 = snell(t, 1.0, n)
        t4 = snell(A - t2, n, 1.0)
        desv.append(np.nan if t4 is None else np.degrees(t + t4 - A))
    desv = np.array(desv)
    plt.plot(np.degrees(th1), desv, color=color, label=f"{lam} nm (mín. {np.nanmin(desv):.2f}°)")
plt.xlabel("ángulo de entrada (°)"); plt.ylabel("desviación (°)")
plt.title("Desviación en un prisma de 60°"); plt.legend(); plt.grid(alpha=0.3); plt.show()

El azul se desvía más que el rojo: alrededor de un grado de diferencia. Es la dispersión cromática que separa la luz blanca en colores, y la que produce la aberración cromática en las lentes. Fíjate también en que la desviación tiene un **mínimo**: Newton lo usó para medir índices de refracción.

## 2. Una lente y su imagen

Trazamos los dos rayos principales de una lente delgada convergente de focal $f$: el que llega paralelo al eje y sale por el foco, y el que pasa por el centro sin desviarse. Donde se cruzan está la imagen. Comprobamos que coincide con $1/d_o + 1/d_i = 1/f$.

In [ ]:
f = 10.0          # focal (cm)
d_o, h_o = 25.0, 3.0   # distancia y altura del objeto (cm)

d_i = 1 / (1 / f - 1 / d_o)
h_i = -h_o * d_i / d_o
print(f"Ecuación de las lentes: d_i = {d_i:.2f} cm, altura de la imagen = {h_i:.2f} cm (negativa: invertida)")

plt.figure(figsize=(10, 4))
plt.axhline(0, color="gray", lw=0.6)
plt.plot([0, 0], [-4.5, 4.5], color="skyblue", lw=6, alpha=0.6)        # la lente
plt.plot([-f, f], [0, 0], "ko", ms=4)                                   # focos
plt.annotate("", xy=(-d_o, h_o), xytext=(-d_o, 0), arrowprops=dict(arrowstyle="->", lw=2))
# rayo 1: paralelo al eje y luego por el foco
plt.plot([-d_o, 0], [h_o, h_o], "r"); plt.plot([0, d_i + 5], [h_o, h_o + (h_i - h_o) / d_i * (d_i + 5)], "r")
# rayo 2: por el centro
plt.plot([-d_o, d_i + 5], [h_o, h_o - h_o / d_o * (d_o + d_i + 5)], "b")
plt.annotate("", xy=(d_i, h_i), xytext=(d_i, 0), arrowprops=dict(arrowstyle="->", lw=2, color="purple"))
plt.title("Dos rayos principales de una lente convergente"); plt.xlabel("cm"); plt.show()

**Ejercicio.** Acerca el objeto a 15 cm, luego a 10 cm (justo en el foco) y luego a 5 cm. ¿Qué le pasa a la imagen? Con $d_o < f$, la ecuación da $d_i$ negativa: es una **imagen virtual**, al mismo lado que el objeto. Así funciona una lupa.

In [ ]:
# Tu código aquí

## 3. Una calculadora de instrumentos

Con el diámetro $D$ y el aumento $M$ (o las focales del objetivo y el ocular) se calcula casi todo:

- captación de luz respecto al ojo: $(D/D_\text{ojo})^2$,
- ganancia en magnitudes: $5\log_{10}(D/D_\text{ojo})$ (lo entenderás del todo en la semana 33),
- pupila de salida: $D/M$,
- resolución de Dawes: $116''/D$ (mm), y la que permite el ojo a ese aumento, $60''/M$.

Usamos una pupila de 6 mm, habitual a partir de los 40 años; con 7 mm salen las cifras del texto de la semana (unas 18 veces más luz para los 8×30). La magnitud límite a simple vista depende mucho del cielo; usamos 5,0 para un cielo suburbano y 6,5 para uno oscuro. Son estimaciones: en la práctica cuentan también la experiencia, la calidad óptica y la altura del objeto.

In [ ]:
def instrumento(nombre, D_mm, M, pupila_ojo=6.0, mag_ojo=5.0):
    luz = (D_mm / pupila_ojo) ** 2
    gan = 5 * np.log10(D_mm / pupila_ojo)
    salida = D_mm / M
    resol = max(116 / D_mm, 60 / M)
    print(f"{nombre:<28} luz ×{luz:6.0f}  +{gan:4.1f} mag  → mag. límite ≈ {mag_ojo + gan:4.1f}"
          f"  pupila de salida {salida:4.1f} mm  resolución ≈ {resol:5.1f}″")

print("Cielo suburbano (magnitud límite 5,0 a simple vista), pupila de 6 mm\n")
instrumento("Ojo", 6, 1)
instrumento("Prismáticos 8×30", 30, 8)
instrumento("Prismáticos 10×50", 50, 10)
instrumento("Refractor 80 mm a 40×", 80, 40)
instrumento("Reflector 130 mm a 65×", 130, 65)
instrumento("Reflector 200 mm a 100×", 200, 100)

Observa la columna de resolución: con pocos aumentos, el límite lo pone tu ojo, no el instrumento. Por eso los prismáticos 8×30 no separan estrellas dobles que un telescopio pequeño separa con facilidad.

**Ejercicio.** Repite la tabla con `mag_ojo=6.5` (un cielo rural oscuro). ¿Qué magnitud límite alcanzan tus prismáticos?

In [ ]:
# Tu código aquí

## 4. El disco de Airy

Por la difracción en la abertura circular, una estrella no se ve como un punto. Su imagen es un disco central rodeado de anillos débiles, con intensidad

$$I(\theta) = I_0\left(\frac{2J_1(x)}{x}\right)^2, \qquad x = \frac{\pi D \sin\theta}{\lambda},$$

donde $J_1$ es una **función de Bessel**, una función especial que aparece en problemas con simetría circular. La calculamos con una integral numérica, $J_1(x) = \frac{1}{\pi}\int_0^\pi \cos(\tau - x\sin\tau)\,d\tau$, para no depender de bibliotecas adicionales. El primer anillo oscuro está en $x = 3{,}832$, es decir, en $\theta \approx 1{,}22\,\lambda/D$.

In [ ]:
def J1(x):
    tau = np.linspace(0, np.pi, 801)
    x = np.atleast_1d(x)[:, None]
    integrando = np.cos(tau - x * np.sin(tau))
    integrar = getattr(np, "trapezoid", None) or np.trapz
    return integrar(integrando, tau, axis=1) / np.pi

lam = 550e-9
theta_arcsec = np.linspace(-12, 12, 1201)
theta = np.radians(theta_arcsec / 3600)
plt.figure(figsize=(10, 4))
for D_mm in [30, 80, 200]:
    x = np.pi * (D_mm / 1000) * np.sin(theta) / lam
    x = np.where(x == 0, 1e-9, x)
    I = (2 * J1(x) / x) ** 2
    plt.plot(theta_arcsec, I, label=f"D = {D_mm} mm (primer anillo oscuro a {np.degrees(1.22 * lam / (D_mm / 1000)) * 3600:.1f}″)")
plt.xlabel("ángulo desde el centro (segundos de arco)"); plt.ylabel("intensidad relativa")
plt.title("Imagen de una estrella: disco de Airy"); plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

Y en dos dimensiones, para unos prismáticos de 30 mm, dos estrellas cada vez más separadas. La del centro, 4,6″, es justo $1{,}22\,\lambda/D$: el **criterio de Rayleigh**, en el que el máximo de una estrella cae sobre el primer anillo oscuro de la otra. Usamos una escala logarítmica de brillo para que se vean los anillos.

In [ ]:
D = 0.030
ang = np.linspace(-15, 15, 241)
X, Y = np.meshgrid(ang, ang)

def airy(dx_arcsec):
    r = np.radians(np.hypot(X - dx_arcsec, Y) / 3600)
    x = np.pi * D * np.sin(r) / lam
    x = np.where(x == 0, 1e-9, x)
    return ((2 * J1(x.ravel()) / x.ravel()) ** 2).reshape(x.shape)

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, sep in zip(axs, [2, 4.6, 8]):
    img = airy(-sep / 2) + airy(sep / 2)
    ax.imshow(np.log10(img + 1e-4), cmap="magma", extent=(-15, 15, -15, 15), vmin=-3, vmax=0.3)
    ax.set_title(f"separación {sep}″"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Dos estrellas vistas con 30 mm de abertura"); plt.tight_layout(); plt.show()

## 5. Experimenta

1. En el prisma, cambia el ángulo a 45° y a 70°. ¿Qué pasa con ángulos de entrada pequeños? (Pista: reflexión total en la segunda cara.)
2. En la calculadora, añade un telescopio con 400 aumentos y 70 mm. ¿Qué pupila de salida tiene? Con menos de 0,5 mm, la imagen se vuelve oscura y borrosa: es un **aumento vacío**.
3. En el disco de Airy, cambia la longitud de onda a 450 y 650 nm. ¿Con qué color se resuelve mejor?

In [ ]:
# Tu experimento aquí